In [ ]:
!pip install -q tensorflow numpy pandas scikit-learn

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Đường dẫn folder ảnh và file csv
image_folder = '/content/drive/MyDrive/Colab Notebooks/Dataset_Melanoma/img'
csv_path = '/content/drive/MyDrive/Colab Notebooks/Dataset_Melanoma/metamelanoma.csv'

# Load ResNet50 pretrained, bỏ lớp cuối (FC), pooling trung bình
model = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_shape=(224,224,3))

### Thư viện sử dụng

Dự án sử dụng ba thành phần chính để xây dựng mô hình phân loại ảnh melanoma.

#### `ResNet50`

- `ResNet50` là mô hình mạng nơ-ron tích chập (CNN) đã được huấn luyện trước trên bộ dữ liệu ImageNet. Trong dự án này, ResNet50 **không được sử dụng để phân loại trực tiếp**, mà đóng vai trò **bộ trích xuất đặc trưng (Feature Extractor)**.

- Sau khi ảnh được đưa vào mô hình, ResNet50 tạo ra một vector đặc trưng có kích thước **2048 chiều**, chứa các thông tin quan trọng về hình dạng, màu sắc và kết cấu của tổn thương da. Các đặc trưng này được sử dụng làm đầu vào cho mô hình học máy ở bước tiếp theo.

#### `preprocess_input`

- Hàm `preprocess_input` thực hiện tiền xử lý ảnh theo đúng chuẩn của ResNet50. Cụ thể, hàm chuyển đổi giá trị điểm ảnh về định dạng mà mô hình đã sử dụng trong quá trình huấn luyện trên ImageNet.

- Việc tiền xử lý giúp đảm bảo dữ liệu đầu vào phù hợp với ResNet50, từ đó nâng cao chất lượng của các đặc trưng được trích xuất.

#### `RandomForestClassifier`

`RandomForestClassifier` là thuật toán học máy được sử dụng để thực hiện **phân loại melanoma**.

Sau khi ResNet50 trích xuất vector đặc trưng từ mỗi ảnh, Random Forest được huấn luyện trên các vector này cùng với nhãn tương ứng. Trong giai đoạn dự đoán, mô hình nhận vector đặc trưng của ảnh mới và đưa ra kết quả thuộc một trong hai lớp:

- Non Melanoma
- Melanoma

Trong toàn bộ pipeline, Random Forest đóng vai trò là **bộ phân loại (Classifier)**, chịu trách nhiệm đưa ra quyết định cuối cùng dựa trên các đặc trưng đã được ResNet50 trích xuất.


In [ ]:
from tqdm import tqdm  # hiển thị progress bar

df = pd.read_csv(csv_path)

features = []  # danh sách để lưu vector
labels = []    # danh sách để lưu nhãn

count_success = 0
count_fail = 0

# Duyệt từng dòng trong dataframe, dùng tqdm để hiển thị thanh tiến trình
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Trích xuất đặc trưng"):
    img_path = os.path.join(image_folder, row['image'] + '.jpg')

    if os.path.exists(img_path):
        #resize
        img = load_img(img_path, target_size=(224,224))

        # Chuyển ảnh sang mảng số
        x = img_to_array(img)

        # Thêm 1 chiều batch size (thành (1,224,224,3)) vì model cần input batch
        x = np.expand_dims(x, axis=0)

        # Chuẩn hóa ảnh theo chuẩn của ResNet50
        x = preprocess_input(x)

        # Dùng model ResNet50 để dự đoán, lấy vector đặc trưng 2048 chiều
        feat = model.predict(x, verbose=0)

        # Flatten vector đặc trưng (loại bỏ batch dimension), rồi lưu vào list
        features.append(feat.flatten())

        # Lưu nhãn melanoma tương ứng (cột 'melanoma' trong CSV)
        labels.append(row['melanoma'])

        count_success += 1
    else:
        print(f'[MISSING] File not found: {img_path}')
        count_fail += 1

# Chuyển list features và labels sang numpy array để dùng cho mô hình ML
features = np.array(features)
labels = np.array(labels)

print(f'Số ảnh đọc thành công: {count_success}')
print(f'Số ảnh bị thiếu: {count_fail}')
print(f'Tổng đặc trưng trích được: {features.shape}')

In [4]:
# Lưu đặc trưng và nhãn
np.savez_compressed('/content/drive/MyDrive/Colab Notebooks/features_labels.npz', features=features, labels=labels)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, stratify=labels, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Tạo confusion matrix từ kết quả dự đoán và nhãn thật
cm = confusion_matrix(y_test, y_pred)

# Vẽ heatmap confusion matrix cho đẹp
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non Melanoma', 'Melanoma'],
            yticklabels=['Non Melanoma', 'Melanoma'])

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
import joblib

# Lưu mô hình vào file
joblib.dump(clf, '/content/drive/MyDrive/Colab Notebooks/random_forest_model.joblib')

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
import os

def predict_single_image_show(image_path):
    if not os.path.exists(image_path):
        print(f"File {image_path} not exit!")
        return

    # Load ảnh và hiển thị
    img = load_img(image_path, target_size=(224, 224))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    # Chuẩn bị ảnh để trích xuất đặc trưng
    x = img_to_array(img)
    x = np.expand_dims(x, axis=0)
    from tensorflow.keras.applications.resnet50 import preprocess_input
    x = preprocess_input(x)

    # Trích xuất đặc trưng và dự đoán
    features = model.predict(x).flatten().reshape(1, -1)
    pred = clf.predict(features)
    prob = clf.predict_proba(features)

    print(f"Predict Label: {pred[0]} with probability: {np.max(prob)*100:.2f}%")

# Gọi hàm dự đoán và hiển thị
predict_single_image_show('/content/drive/MyDrive/Colab Notebooks/Dataset/1200px-Melanoma.jpg')